# 04_yolo26n_comparison

E1: modern detector under the same protocol

Status: prepared; results require execution.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

if Path("/kaggle/input").exists():
    bundles = sorted(Path("/kaggle/input").rglob("project.zip"))
    expanded = [p.parent for p in Path("/kaggle/input").rglob("pyproject.toml")
                if (p.parent / "src/configs/sources.json").is_file()]
    ROOT = Path("/kaggle/working/dentalselfie")
    ROOT.mkdir(exist_ok=True)
    if len(expanded) == 1 and not bundles:
        shutil.copytree(expanded[0], ROOT, dirs_exist_ok=True)
    elif not expanded and len(bundles) == 1:
        with zipfile.ZipFile(bundles[0]) as archive:
            for member in archive.infolist():
                target = (ROOT / member.filename).resolve()
                if not target.is_relative_to(ROOT.resolve()) or ":" in member.filename:
                    raise ValueError("Unsafe code-bundle path")
            archive.extractall(ROOT)
    else:
        raise RuntimeError("Attach exactly one DentalSelfie Code dataset")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT),
                    "-r", str(ROOT / "requirements-kaggle.txt")], check=True)
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/configs/sources.json").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Open this notebook inside the DentalSelfie project")
os.chdir(ROOT)
print("Project:", ROOT)


In [ ]:
if Path("/kaggle/input").exists():
    candidates = [p.parent for p in Path("/kaggle/input").rglob("sources.json")
                  if (p.parent / "caries/data.yaml").is_file()
                  and (p.parent / "gum/data.yaml").is_file()]
    if not candidates:
        archives = sorted(Path("/kaggle/input").rglob("prepared.zip"))
        if len(archives) == 1:
            from dentalselfie.audit import safe_extract
            destination = Path("/kaggle/working/prepared_data")
            if not destination.exists():
                safe_extract(archives[0], destination)
            candidates = [destination]
    if len(candidates) != 1:
        raise RuntimeError("Attach exactly one prepared DentalSelfie Data dataset")
    DATA_ROOT = candidates[0]
else:
    raise RuntimeError("Heavy training runs on Kaggle T4 x2; local data tooling has its own notebooks")
print("Prepared dataset:", DATA_ROOT)


In [ ]:
RUN_TRAINING = True  # Agent switches to True only after prerequisites and smoke run pass.
SMOKE = True  # Three epochs per task measure throughput; excluded from research results.
SEED = 42
MAX_SECONDS = 1800 if SMOKE else 39600  # 11h training leaves room for setup and finalization.
subprocess.run(["nvidia-smi"], check=True)
if RUN_TRAINING:
    command = [sys.executable, "-m", "dentalselfie.run_pair", "--experiment", "E1",
               "--config", str(ROOT / "src/configs/experiments.json"), "--data-root", str(DATA_ROOT),
               "--output", "/kaggle/working/experiments", "--seed", str(SEED),
               "--max-seconds", str(MAX_SECONDS)]
    if SMOKE:
        command.append("--smoke")
    subprocess.run(command, check=True)
else:
    print("Not trained. This notebook is a prepared entry point, with no fabricated outputs.")


Exit gate: compare against E0 using identical splits, input size and declared training budget. Do not imply that stock YOLO26 reproduces custom dental paper modules.

## Execution evidence and throughput

Smoke runs use three epochs at the configured batch size, AMP and workers. They are excluded from final comparisons. Full runs use validation for checkpoint selection; test is never evaluated here. Save the whole experiments output to retain logs, configuration, dependency freeze, curves, metrics and resumable checkpoints.

In [ ]:
import json
import pandas as pd
from IPython.display import display

output = Path("/kaggle/working/experiments")
rows = []
for path in sorted(output.glob("*.result.json")):
    item = json.loads(path.read_text())
    rows.append({"track": path.stem, "status": item["status"], "smoke_only": item["smoke_only"],
                 "epochs": item["epochs_completed"], "seconds": item["elapsed_seconds"],
                 **item["metrics"]})
if rows:
    summary = pd.DataFrame(rows)
    display(summary)
    summary.to_csv(output / "session_summary.csv", index=False)
    for path in sorted(output.glob("*/results.csv")):
        history = pd.read_csv(path)
        display(history)
        if len(history) > 1:
            steady_epoch = history["time"].diff().dropna().median()
            print(path.parent.name, "median post-first epoch seconds:", round(steady_epoch, 2),
                  "estimated 100-epoch hours:", round(steady_epoch * 100 / 3600, 2))
    display(summary.select_dtypes("number"))
else:
    print("No completed worker results yet; this is not an experiment result.")


## Checkpoint and session protocol

One task runs on each T4. After every epoch, resume.pt retains the optimizer, EMA and scaler before finalization can strip last.pt. Separate state records scheduler and process RNG states; data-loader worker/prefetch states are not serialized, so resume is not bitwise identical. A wall-clock guard stops at an epoch boundary after 11 hours; the Kaggle session timeout is 12 hours. Setup, final validation and output publication need the remaining margin. An unusually long epoch can exceed this target; smoke measurements must precede full runs.

In [ ]:
import inspect
from IPython.display import Code, display
from dentalselfie.checkpoints import CheckpointGuard
import dentalselfie.train as worker
display(Code(inspect.getsource(worker), language="python"))
display(Code(inspect.getsource(CheckpointGuard), language="python"))